[Guide Index](README.md)

# 02b The Oxigraph-backed graph (`StarLayerGraph` / `StarLayerDataset`)

`StarLayerGraph` (one graph) and `StarLayerDataset` (a set of named graphs) each hold a `pyoxigraph.Store`, the Oxigraph database.

`StarLayerGraph` and `StarLayerDataset` have three kinds of methods:

- rdflib-style methods (`add`, `remove`, `triples`, `parse`, `serialize`, `query`, `update`, `bind`).
- other methods StarLayer adds (`infer`, `profile`, `skolemize`, ...).
- the Oxigraph store's own methods (`load`, `quads_for_pattern`, `named_graphs`, ...), reached through `.ox`.

Every method of the Oxigraph store can be called through `.ox`, for example `g.ox.load(...)`, on the graph or the dataset. The methods can also be called on the store directly, reachable through `.store`. The preferred way is `.ox`. It records a change after a write, which is useful when StarLayer maintains caches of store data.

The Oxigraph methods are documented in [pyoxigraph](https://pyoxigraph.readthedocs.io/).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones. Section 2 needs the OWL reasoners (`pip install starlayer-reasoning`, see [2.e](02e-graphs-inferencing.ipynb)) and section 3 needs rdflib (`pip install "starlayer[rdflib]"`). 

## 1. Oxigraph's methods through `.ox`

All the graphs of a dataset share one store. A `StarLayerGraph` is one graph of it, and `.store` is the whole store. There are three ways to load data into a graph. Here `people` gets one triple each way:

1. **rdflib style**: `parse`.
2. **The store's `load`, called through `.ox`**: Oxigraph's own method, with Oxigraph's arguments.
3. **The same `load`, called on `.store`**: the same arguments and the same result.


In [ ]:
import pyoxigraph as ox

from starlayer.graph import Namespace, StarLayerDataset

ex = Namespace("http://example.org/")
ds = StarLayerDataset()
people = ds.graph(ex.people)

# 1. rdflib style
people.parse(data="@prefix ex: <http://example.org/> . ex:ann ex:knows ex:bob .", format="turtle")
# 2. the store's load, called through .ox: the graph is named, as Oxigraph requires
people.ox.load(b"@prefix ex: <http://example.org/> . ex:ann ex:knows ex:cara .", ox.RdfFormat.TURTLE, to_graph=ex.people)
# 3. the same load, called on .store
people.store.load(b"@prefix ex: <http://example.org/> . ex:ann ex:knows ex:dan .", ox.RdfFormat.TURTLE, to_graph=ex.people)

print("people:", len(people))

people: 3


Add a second graph, `places`, to the same store. A graph acts only on itself: asking `people` to load into `places` is a `ValueError`.

Methods about the store as a whole, such as `named_graphs`, act on the whole store, also when called on a graph, so `people.ox.named_graphs()` lists both graphs.

In [2]:
places = ds.graph(ex.places)
places.ox.load(b"@prefix ex: <http://example.org/> . ex:oslo ex:in ex:norway .", ox.RdfFormat.TURTLE, to_graph=ex.places)
print("people:", len(people), "places:", len(places), "| the store holds", len(people.store), "quads, shared:", people.store is places.store)

try:
    people.ox.load(b"@prefix ex: <http://example.org/> . ex:x ex:p ex:y .", ox.RdfFormat.TURTLE, to_graph=ex.places)
except ValueError as error:
    print("ValueError:", error)

print("named graphs, from a graph:", sorted(str(n) for n in people.ox.named_graphs()))

people: 3 places: 1 | the store holds 4 quads, shared: True
ValueError: this StarLayerGraph holds graph <http://example.org/people>; ox.load() cannot act on graph <http://example.org/places>
named graphs, from a graph: ['<http://example.org/people>', '<http://example.org/places>']


## 2. Write through the graph, not `.store`: the caching warning

StarLayer keeps some results between calls, and uses the record of changes to know when they are out of date. For example, StarLayer caches entailed triples of `query(entailment=..., recompute="onChange")` and discards the cache when the graph changes. A write made directly on `.store` is not registered as a change, so those results may go stale. Query entailment and the `infer=` option are covered in [3.a](03c-sparql-entailment.ipynb), section 3.

The example below caches the entailed graph of an OWL 2 RL query, then adds a fact directly on the store. The cache becomes stale. Adding the same fact with `g.add` records the change, and the subsequent query refreshes the cache. `g.ox.add(quad)` is Oxigraph's own add, which takes a quad, and it records the change.  

In [ ]:
import pyoxigraph as ox

from starlayer.graph import ENTAILMENT, RDF, Namespace, StarLayerGraph

ex = Namespace("http://example.org/")
g = StarLayerGraph()
g.parse(
    data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

    ex:car rdfs:subClassOf ex:vehicle .
    """,
    format="turtle",
)

# is myCar a vehicle?
QUERY = """
    PREFIX ex: <http://example.org/>
    ASK { ex:myCar a ex:vehicle }
"""

# run the query
def my_car_is_a_vehicle():
    return g.query(QUERY, entailment=ENTAILMENT["OWL-RDF-Based"], recompute="onChange").askAnswer


print("myCar is a vehicle:", my_car_is_a_vehicle())                                          # the entailed graph is now cached
g.store.add(ox.Quad(ex.myCar, RDF.type, ex.car, ox.DefaultGraph()))           # Oxigraph's add on .store: StarLayer is not told
print("after a write on .store:", my_car_is_a_vehicle(), "(stale)")
g.add((ex.myCar, RDF.type, ex.car))                                           # the standard add: StarLayer records the change
print("after a write through the graph:", my_car_is_a_vehicle())                         # the change causes the next query to refresh the cache

myCar is a vehicle: False
after a write on .store: False (stale)
after a write through the graph: True


## 3. Converting to and from rdflib

A `StarLayerGraph` is not an `rdflib.Graph`. The availale methods have the same shape, but the terms that are returned are pyoxigraph terms. rdflib terms and graphs are not accepted. To convert a whole graph, use `StarLayerGraph.from_rdflib()` and `to_rdflib()`. Both need rdflib (`pip install "starlayer[rdflib]"`), which StarLayer imports when a conversion requires it.

The example parses a graph in rdflib, converts it, adds an annotation (RDF 1.2: a reifier, `ex:claim`, that says something about a triple), and converts it back. rdflib cannot hold a triple term, so the annotation comes back in the RDF 1.1 basic encoding, the same one `serialize(rdf_version="1.1")` writes.

In [ ]:
import rdflib
# an rdflib graph
rdf_graph_1 = rdflib.Graph()
rdf_graph_1.parse(
    data="""
    @prefix ex: <http://example.org/> .

    ex:ann ex:knows ex:bob .
    """,
    format="turtle",
)
print("1. the rdflib graph\n" + rdf_graph_1.serialize(format="turtle"))

# convert it to a StarLayerGraph (an Oxigraph store)
g = StarLayerGraph.from_rdflib(rdf_graph_1)
g.add_reification(ex.claim, ox.Triple(ex.ann, ex.knows, ex.bob))     # ex:claim is an annotation on the triple
g.add((ex.claim, ex.since, ox.Literal("2020")))

print("2. in StarLayer, with the annotation\n" + g.serialize(format="turtle", only_used=True, pretty=True))

# converted back to rdflib using RDF 1.1 basic encoding.
rdf_graph_2 = g.to_rdflib()
print("3. back in rdflib\n" + rdf_graph_2.serialize(format="turtle"))

1. the rdflib graph
@prefix ex: <http://example.org/> .

ex:ann ex:knows ex:bob .


2. in StarLayer, with the annotation
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:ann ex:knows ex:bob ~ ex:claim {| ex:since "2020" |} .

3. back in rdflib
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:since "2020" ;
    rdf:reifies [ a rdf:PropositionForm ;
            rdf:propositionFormObject ex:bob ;
            rdf:propositionFormPredicate ex:knows ;
            rdf:propositionFormSubject ex:ann ] .

ex:ann ex:knows ex:bob .


